# Algoritmos de ordenamiento avanzados

La transición de algoritmos de ordenamiento cuadráticos $\mathcal{O}(n^2)$ a logarítmico-lineales $\mathcal{O}(n \log n)$ no es una mera curiosidad matemática; es una exigencia absoluta en la ingeniería de software escalable. A medida que el volumen de datos crece, la disparidad en el tiempo de ejecución entre estas dos clases de complejidad se vuelve insalvable para cualquier arquitectura de hardware actual.

En la siguiente celda se quiere ilustrar empíricamente esta divergencia asintótica.

In [1]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <random>
#include <chrono>

// Implementación ingenua O(n^2) para propósitos de demostración
void bubble_sort(std::vector<int>& arr) {
    bool swapped;
    size_t n = arr.size();
    do {
        swapped = false;
        for (size_t i = 1; i < n; ++i) {
            if (arr[i - 1] > arr[i]) {
                std::swap(arr[i - 1], arr[i]);
                swapped = true;
            }
        }
        --n; // Optimización marginal: el último elemento ya está en su lugar
    } while (swapped);
}

void ejemplo_01() {
    constexpr size_t DATA_SIZE = 50000; // Un tamaño moderado para evidenciar el cuello de botella
    std::vector<int> data_n2(DATA_SIZE);
    
    // Generación de datos
    std::mt19937 gen(42); // Semilla fija para reproducibilidad determinista
    std::uniform_int_distribution<> dis(1, 100000);
    std::generate(data_n2.begin(), data_n2.end(), [&]() { return dis(gen); });
    
    // Copia exacta para el algoritmo O(n log n)
    std::vector<int> data_nlogn = data_n2;

    // Benchmark O(n^2)
    auto start_n2 = std::chrono::high_resolution_clock::now();
    bubble_sort(data_n2);
    auto end_n2 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_n2 = end_n2 - start_n2;

    // Benchmark O(n log n)
    auto start_nlogn = std::chrono::high_resolution_clock::now();
    std::sort(data_nlogn.begin(), data_nlogn.end());
    auto end_nlogn = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_nlogn = end_nlogn - start_nlogn;

    std::cout << "Rendimiento para N = " << DATA_SIZE << " elementos:\n";
    std::cout << "Complejidad O(n^2)     : " << duration_n2.count() << " ms\n";
    std::cout << "Complejidad O(n log n) : " << duration_nlogn.count() << " ms\n";
    std::cout << "Factor de aceleración  : " << duration_n2.count() / duration_nlogn.count() << "x\n";
}

ejemplo_01();

Rendimiento para N = 50000 elementos:
Complejidad O(n^2)     : 13181.5 ms
Complejidad O(n log n) : 12.2863 ms
Factor de aceleración  : 1072.87x


* Utilizamos std::chrono::high_resolution_clock para obtener marcas de tiempo con precisión de nanosegundos, evitando la imprecisión inherente de utilidades legacy como clock().
  
* Para $N = 50,000$, un algoritmo $\mathcal{O}(n^2)$ requiere del orden de $2.5 \times 10^9$ operaciones, mientras que un algoritmo $\mathcal{O}(n \log n)$ requiere aproximadamente $7.8 \times 10^5$. El factor de aceleración empírico superará los tres órdenes de magnitud.

* La imposibilidad de escalar algoritmos iterativos anidados justifica la necesidad de adoptar estrategias de partición recursiva del espacio de búsqueda. Merge Sort y Quick Sort encarnan esta filosofía matemática: al segmentar un problema de tamaño $N$ en subproblemas fraccionales, el número de niveles de recursión queda acotado por $\log_2(N)$, resolviendo la limitación fundamental observada en este benchmark.

# Implementación de mergesort

In [2]:
#include <iostream>
#include <vector>

// Procedimiento Combinar (MERGE)
template <typename T>
void merge(std::vector<T>& arr, size_t left, size_t mid, size_t right) {
    size_t n1 = mid - left + 1;
    size_t n2 = right - mid;

    // Asignación de memoria en el heap (overhead espacial)
    std::vector<T> L(n1);
    std::vector<T> R(n2);

    // Copia de los subarreglos
    for (size_t i = 0; i < n1; ++i)
        L[i] = arr[left + i];
    for (size_t j = 0; j < n2; ++j)
        R[j] = arr[mid + 1 + j];

    // Índices de rastreo
    size_t i = 0, j = 0, k = left;

    // Fusión manteniendo el orden y la estabilidad
    while (i < n1 && j < n2) {
        if (L[i] <= R[j]) { 
            arr[k] = L[i];
            i++;
        } else {
            arr[k] = R[j];
            j++;
        }
        k++;
    }

    // Volcado de los elementos residuales
    while (i < n1) { arr[k++] = L[i++]; }
    while (j < n2) { arr[k++] = R[j++]; }
}

// Procedimiento Recursivo Principal (MERGE-SORT)
template <typename T>
void merge_sort(std::vector<T>& arr, size_t left, size_t right) {
    if (left >= right) return; // Caso base: n <= 1

    // Cálculo seguro del punto medio para evitar desbordamiento (Integer Overflow)
    size_t mid = left + (right - left) / 2;

    // Fase de Conquista
    merge_sort(arr, left, mid);
    merge_sort(arr, mid + 1, right);

    // Fase de Combinación
    merge(arr, left, mid, right);
}

// Función envoltura (Wrapper) para interfaz limpia
template <typename T>
void merge_sort(std::vector<T>& arr) {
    if (!arr.empty()) {
        merge_sort(arr, 0, arr.size() - 1);
    }
}


void ejemplo_02() {
    constexpr size_t DATA_SIZE = 50000; // Un tamaño moderado para evidenciar el cuello de botella
    std::vector<int> data_n2(DATA_SIZE);
    
    // Generación de datos
    std::mt19937 gen(42); // Semilla fija para reproducibilidad determinista
    std::uniform_int_distribution<> dis(1, 100000);
    std::generate(data_n2.begin(), data_n2.end(), [&]() { return dis(gen); });
    
    // Copia exacta para el algoritmo O(n log n)
    std::vector<int> data_nlogn = data_n2;

    // Benchmark O(n^2)
    auto start_n2 = std::chrono::high_resolution_clock::now();
    merge_sort(data_n2);
    auto end_n2 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_n2 = end_n2 - start_n2;

    // Benchmark O(n log n)
    auto start_nlogn = std::chrono::high_resolution_clock::now();
    std::sort(data_nlogn.begin(), data_nlogn.end());
    auto end_nlogn = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_nlogn = end_nlogn - start_nlogn;

    std::cout << "Rendimiento para N = " << DATA_SIZE << " elementos:\n";
    std::cout << "Complejidad O(n log n) - MERGESORT     : " << duration_n2.count() << " ms\n";
    std::cout << "Complejidad O(n log n) - std::sort     : " << duration_nlogn.count() << " ms\n";
    std::cout << "Factor de aceleración  : " << duration_n2.count() / duration_nlogn.count() << "x\n";
}

ejemplo_02();

Rendimiento para N = 50000 elementos:
Complejidad O(n log n) - MERGESORT     : 29.8186 ms
Complejidad O(n log n) - std::sort     : 12.3677 ms
Factor de aceleración  : 2.41101x


* ***Prevención de Desbordamiento***: En lugar del ingenuo `(left + right) / 2`, la expresión `left + (right - left) / 2` garantiza seguridad tipo memory-safe al trabajar con índices de escala masiva en arreglos de gran tamaño.

* ***Estabilidad (Stability)***: Observe el uso del operador `<=` en la instrucción `if (L[i] <= R[j])`. Esta es una decisión crítica: garantiza que, si dos llaves (keys) son idénticas, el elemento del subarreglo izquierdo preserve su posición original respecto al elemento del derecho. Esto define a Merge Sort como un ordenamiento ***estable***.

## Costo asintotico 

Podemos modelar el costo asintótico en el peor caso a través de una ecuación de recurrencia $T(n)$.
* ***División*** ($D(n)$): El cálculo del punto medio es una simple operación aritmética que toma tiempo constante, $\Theta(1)$.

* ***Conquista*** ($aT(n/b)$): Se efectúan 2 llamadas recursivas sobre subarreglos de tamaño $n/2$, lo cual impone un costo de $2T(n/2)$.

* ***Combinación*** ($C(n)$): La subrutina MERGE itera sobre un total de $n$ elementos a través de tres ciclos while, ejecutando asignaciones simples que consumen tiempo lineal, por lo que $C(n) = \Theta(n)$.La recurrencia resultante es:

$$T(n) = \begin{cases} \Theta(1) & \text{si } n = 1 \\ 2T(n/2) + \Theta(n) & \text{si } n > 1 \end{cases}$$

Aplicando el método del árbol de recursión (*Recursion-tree method, Capítulo 4 de Cormen, 2022*), observamos que el árbol posee una profundidad exacta de $\log_2(n) + 1$. El costo computacional sumado horizontalmente por cada nivel del árbol es exactamente $cn$ (donde $c$ es una constante intrínseca de la máquina).

* ***Complejidad Temporal*** (Uniforme para Mejor, Peor y Caso Promedio): $\Theta(n \log n)$. Esto comprueba la optimalidad del algoritmo bajo el modelo de árboles de decisión.

*  ***Complejidad Espacial** : Dado que la función merge instancia dinámicamente dos vectores `L` y `R`, la complejidad espacial totaliza $\Theta(n)$. Este algoritmo no opera in-place, limitando severamente su utilización en sistemas con recursos de memoria volátil constreñidos.

# Implementación de Quicksort

In [3]:
#include <iostream>
#include <vector>
#include <utility> // para std::swap
#include <random>

// Subrutina de Partición (Esquema de Lomuto)
template <typename T>
size_t partition(std::vector<T>& arr, size_t low, size_t high) {
    // Selección del pivote (tomamos el último elemento temporalmente)
    T pivot = arr[high];
    
    // i representa el límite superior de los elementos menores o iguales al pivote
    size_t i = low; 

    for (size_t j = low; j < high; ++j) {
        if (arr[j] <= pivot) {
            std::swap(arr[i], arr[j]);
            i++;
        }
    }
    // Posicionar el pivote en su ubicación final y correcta
    std::swap(arr[i], arr[high]);
    return i;
}

// Subrutina principal recursiva
template <typename T>
void quick_sort_recursive(std::vector<T>& arr, size_t low, size_t high) {
    if (low < high) {
        // q es el índice de partición; arr[q] ya está en su posición final
        size_t q = partition(arr, low, high);

        // Llamadas recursivas para ordenar los subarreglos
        // Se utiliza la condición para evitar underflow en size_t si q == 0
        if (q > 0) {
            quick_sort_recursive(arr, low, q - 1);
        }
        quick_sort_recursive(arr, q + 1, high);
    }
}

// Interfaz pública
template <typename T>
void quick_sort(std::vector<T>& arr) {
    if (!arr.empty()) {
        quick_sort_recursive(arr, 0, arr.size() - 1);
    }
}

void ejemplo_03() {
    constexpr size_t DATA_SIZE = 50000; // Un tamaño moderado para evidenciar el cuello de botella
    std::vector<int> data_n2(DATA_SIZE);
    
    // Generación de datos
    std::mt19937 gen(42); // Semilla fija para reproducibilidad determinista
    std::uniform_int_distribution<> dis(1, 100000);
    std::generate(data_n2.begin(), data_n2.end(), [&]() { return dis(gen); });
    
    // Copia exacta para el algoritmo O(n log n)
    std::vector<int> data_nlogn = data_n2;

    // Benchmark O(n^2)
    auto start_n2 = std::chrono::high_resolution_clock::now();
    quick_sort(data_n2);
    auto end_n2 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_n2 = end_n2 - start_n2;

    // Benchmark O(n log n)
    auto start_nlogn = std::chrono::high_resolution_clock::now();
    std::sort(data_nlogn.begin(), data_nlogn.end());
    auto end_nlogn = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_nlogn = end_nlogn - start_nlogn;

    std::cout << "Rendimiento para N = " << DATA_SIZE << " elementos:\n";
    std::cout << "Complejidad O(n log n) - QUICKSORT     : " << duration_n2.count() << " ms\n";
    std::cout << "Complejidad O(n log n) - std::sort     : " << duration_nlogn.count() << " ms\n";
    std::cout << "Factor de aceleración  : " << duration_n2.count() / duration_nlogn.count() << "x\n";
}

ejemplo_03();

Rendimiento para N = 50000 elementos:
Complejidad O(n log n) - QUICKSORT     : 9.57072 ms
Complejidad O(n log n) - std::sort     : 12.3825 ms
Factor de aceleración  : 0.77292x


El análisis matemático de Quick Sort difiere sustancialmente de Merge Sort debido a que el tamaño de los subarreglos resultantes de la partición depende directamente de la naturaleza de los datos de entrada (Cormen, Sección 7.2, págs. 186-190).

* ***El Peor Caso*** ($\mathcal{O}(n^2)$):Ocurre cuando la rutina de partición produce un subarreglo con $n-1$ elementos y otro con $0$ elementos iterativamente. Bajo el esquema de Lomuto (pivote en $A[r]$), esto sucede si el arreglo ya está ordenado (o inversamente ordenado). La ecuación de recurrencia es:
$$T(n) = T(n-1) + T(0) + \Theta(n) = T(n-1) + \Theta(n) \implies \Theta(n^2)$$
Esta degradación es inaceptable en sistemas de producción.

* ***El Mejor Caso*** ($\mathcal{O}(n \log n)$):Si la partición divide el arreglo en dos mitades exactamente iguales (o de tamaño $\lfloor n/2 \rfloor$ y $\lceil n/2 \rceil - 1$), la recurrencia es idéntica a la de Merge Sort:
$$T(n) = 2T(n/2) + \Theta(n) \implies \Theta(n \log n)$$

* ***El Caso Promedio y la Optimización Estocástica*** (Randomized Quick Sort):En la práctica, cualquier división que sea proporcional (por ejemplo, 9-a-1) genera un árbol de recursión de profundidad logarítmica $\mathcal{O}(\log n)$, manteniendo la cota de tiempo en $\mathcal{O}(n \log n)$ con constantes ocultas significativamente menores que las de Merge Sort. Para blindar el algoritmo contra el peor caso, Cormen (Sección 7.3, pág. 190) introduce Randomized-Partition. Consiste en intercambiar A[r] con un elemento elegido uniformemente al azar de A[p...r] antes de particionar. Esto garantiza que la complejidad esperada sea $\mathcal{O}(n \log n)$ independientemente de la distribución inicial de la entrada.

* ***Complejidad Espacial*** ($\mathcal{O}(\log n)$ a $\mathcal{O}(n)$):Aunque Quick Sort ordena in-place (no asigna memoria dinámica como MERGE), el árbol de llamadas recursivas consume espacio en el Call Stack. En el peor caso, la profundidad del árbol es $n$, requiriendo espacio $\mathcal{O}(n)$. En el mejor caso o caso promedio, requiere $\mathcal{O}(\log n)$. (En la industria, esto se mitiga ordenando siempre el subarreglo más pequeño primero mediante recursión de cola).

# Criterios Arquitectónicos y Selección de Algoritmos

Tanto Merge Sort como Quick Sort comparten una complejidad esperada de $\Theta(n \log n)$, la decisión de ingeniería sobre cuál implementar en un entorno de producción no se basa exclusivamente en el análisis asintótico, sino en la arquitectura de hardware subyacente y los requerimientos de integridad de los datos.Existen dos factores críticos de decisión:
1. ***Localidad de Referencia (Caché)***: Quick Sort opera in-place. Al realizar intercambios dentro del mismo bloque de memoria contigua, maximiza los aciertos de caché (cache hits) en las líneas L1/L2 del procesador. Merge Sort, al escribir en arreglos auxiliares, genera un patrón de acceso a memoria que provoca fallos de caché (cache misses) constantes, degradando el rendimiento empírico.
2. ***Estabilidad Algorítmica***: Como se discutió, Merge Sort es inherentemente estable (preserva el orden relativo de elementos con llaves idénticas). Quick Sort, debido a los intercambios no adyacentes en la subrutina de partición, es inestable.Para ilustrar este criterio de decisión en código de producción, examinaremos un despachador en C++ que selecciona la estrategia de ordenamiento óptima basada en el requerimiento del sistema.

## `std::stable_sort` vs `std::sort`: 
La biblioteca estándar de C++ abstrae los algoritmos subyacentes, pero sus garantías de complejidad dictan las implementaciones. 
* `std::stable_sort` asigna memoria dinámica para ejecutar un Merge Sort (o recurre a un algoritmo en sitio $\mathcal{O}(n \log^2 n)$ si la memoria falla).

* `std::sort` utiliza IntroSort (Quick Sort como motor principal).

In [4]:
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

// Estructura de datos que representa una transacción o registro en base de datos
struct Transaction {
    int severity;      // Llave primaria de ordenamiento
    int timestamp;     // Llave secundaria (orden temporal original)
    std::string id;
};

// Utilidad para imprimir el estado
void print_transactions(const std::vector<Transaction>& txs) {
    for (const auto& tx : txs) {
        std::cout << "[Sev: " << tx.severity << ", Time: " << tx.timestamp 
                  << ", ID: " << tx.id << "]\n";
    }
    std::cout << "-----------------------------------\n";
}

// Despachador arquitectónico
void sort_transactions(std::vector<Transaction>& txs, bool require_stability) {
    // Definición de la heurística de comparación (Lambda C++11/14+)
    auto comparator = [](const Transaction& a, const Transaction& b) {
        return a.severity < b.severity; // Ordenamos exclusivamente por severidad
    };

    if (require_stability) {
        // Implementación subyacente: Merge Sort (o derivado). 
        // Garantiza O(n log n) en el peor caso. Espacio auxiliar O(n).
        std::cout << "Ejecutando Ordenamiento Estable (Merge Sort):\n";
        std::stable_sort(txs.begin(), txs.end(), comparator);
    } else {
        // Implementación subyacente: IntroSort (Quick Sort híbrido).
        // Optimizado para caché. Espacio auxiliar O(log n).
        std::cout << "Ejecutando Ordenamiento In-Place (Quick Sort):\n";
        std::sort(txs.begin(), txs.end(), comparator);
    }
}

void ejemplo_04() {
    // Vector de 20 elementos para evadir el umbral de Insertion Sort (n < 16)
    // Esto obliga a std::sort a utilizar la rutina de partición de Quick Sort.
    // Además, se inyecta una alta densidad de colisiones en la severidad '3' y '2'.
    std::vector<Transaction> data1 = {
        {3, 100, "TX_01"}, {1, 105, "TX_02"}, {3, 110, "TX_03"}, {2, 115, "TX_04"},
        {3, 120, "TX_05"}, {1, 125, "TX_06"}, {3, 130, "TX_07"}, {2, 135, "TX_08"},
        {3, 140, "TX_09"}, {3, 145, "TX_10"}, {3, 150, "TX_11"}, {1, 155, "TX_12"},
        {2, 160, "TX_13"}, {3, 165, "TX_14"}, {3, 170, "TX_15"}, {1, 175, "TX_16"},
        {3, 180, "TX_17"}, {2, 185, "TX_18"}, {3, 190, "TX_19"}, {3, 195, "TX_20"}
    };
    
    // Copia profunda para contrastar el determinismo
    std::vector<Transaction> data2 = data1;

    // Prueba 1: Sin requerimiento de estabilidad (Quick Sort / IntroSort)
    sort_transactions(data1, false);
    print_transactions(data1); 
    // Nota: Al observar las transacciones con severidad 3, notará que sus identificadores 
    // (TX_01, TX_03, TX_05...) están completamente desordenados temporalmente. 
    // La inestabilidad de Quick Sort se ha manifestado.

    // Prueba 2: Requerimiento de estabilidad estricto (Merge Sort)
    sort_transactions(data2, true);
    print_transactions(data2);
    // Nota: Aquí, todas las transacciones de severidad 3 mantienen rigurosamente 
    // su orden ascendente de 'timestamp' (TX_01 -> TX_03 -> TX_05...).
}

ejemplo_04();

Ejecutando Ordenamiento In-Place (Quick Sort):
[Sev: 1, Time: 105, ID: TX_02]
[Sev: 1, Time: 125, ID: TX_06]
[Sev: 1, Time: 175, ID: TX_16]
[Sev: 1, Time: 155, ID: TX_12]
[Sev: 2, Time: 115, ID: TX_04]
[Sev: 2, Time: 185, ID: TX_18]
[Sev: 2, Time: 135, ID: TX_08]
[Sev: 2, Time: 160, ID: TX_13]
[Sev: 3, Time: 195, ID: TX_20]
[Sev: 3, Time: 100, ID: TX_01]
[Sev: 3, Time: 190, ID: TX_19]
[Sev: 3, Time: 180, ID: TX_17]
[Sev: 3, Time: 170, ID: TX_15]
[Sev: 3, Time: 165, ID: TX_14]
[Sev: 3, Time: 150, ID: TX_11]
[Sev: 3, Time: 145, ID: TX_10]
[Sev: 3, Time: 140, ID: TX_09]
[Sev: 3, Time: 130, ID: TX_07]
[Sev: 3, Time: 120, ID: TX_05]
[Sev: 3, Time: 110, ID: TX_03]
-----------------------------------
Ejecutando Ordenamiento Estable (Merge Sort):
[Sev: 1, Time: 105, ID: TX_02]
[Sev: 1, Time: 125, ID: TX_06]
[Sev: 1, Time: 155, ID: TX_12]
[Sev: 1, Time: 175, ID: TX_16]
[Sev: 2, Time: 115, ID: TX_04]
[Sev: 2, Time: 135, ID: TX_08]
[Sev: 2, Time: 160, ID: TX_13]
[Sev: 2, Time: 185, ID: TX_18]
[Se